In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import heapq
import seaborn as sns

from environment.environment import GraphWorldMFG_MultiGroup
from trainer.amid_trainer_graph import GraphEdgeMFG_Trainer
from solver.solver import solve_multigroup, GraphMFG_OMD_EdgeSolver_MultiGroup
from visualization.visualizationh import plot_heatmap, compute_exploitability_multigroup, plot_losses, plot_losses_line

In [2]:
import os
from pathlib import Path
import yaml
import numpy as np
import torch

# 1. Configuration Loader
def load_config(config_path="config.yaml"):
    """Load configuration safely from a YAML file relative to working directory."""
    notebook_dir = Path(os.getcwd())
    config_file = notebook_dir / config_path
    
    with open(config_file, 'r') as f:
        config = yaml.safe_load(f)
    return config

# 2. Graph Environment Factory Pattern
def create_graph_mfg_from_config(config):
    """Factory function initializing the Graph MFG environment directly from your config."""
    device = config.get("device", "cuda" if torch.cuda.is_available() else "cpu")
    print(f"Target Device: {device}")

    trainer_cfg = config["trainer"]
    
    # Extract Graph Configuration
    graph_cfg = config["graph"]
    num_nodes = graph_cfg["num_nodes"]
    
    # CRITICAL: Extract the list of lists matrix and convert to a PyTorch Tensor
    raw_matrix = graph_cfg["adjacency_matrix"]
    adjacency_matrix = torch.tensor(raw_matrix, dtype=torch.float32, device=device)
    
    # Process Group Data (Sinks and Sources are flat integers here, not grid tuples)
    groups = []
    for g in config["groups"]:
        groups.append({
            "source": int(g["source"]),
            "sink": int(g["sink"]),
            "mass": float(g["mass"])
        })
        
    # Instantiate the Graph World Environment
    # (Matches your 'GraphWorldMFG_MultiGroup' class structure)
    env = GraphWorldMFG_MultiGroup(
        num_nodes=num_nodes,
        groups=groups,
        adjacency_matrix=adjacency_matrix,
        device=device
    )
    
    # Create solvers for each group
    solver_cfg = config["solver"]
    waiting_time = solver_cfg.get("waiting_time", 0)
    solvers = [
        GraphMFG_OMD_EdgeSolver_MultiGroup(
            env=env,
            group_idx=k,
            eta=solver_cfg["eta"],
            tau=solver_cfg["tau"],
            T=solver_cfg["T"],
            alpha=solver_cfg["alpha"],
            H=solver_cfg.get("H", None)
        )
        for k in range(env.K)
    ]

    trainer = GraphEdgeMFG_Trainer(env, solvers, leader_lr=trainer_cfg["leader_lr"])
    
    return env, solvers, trainer, config

In [3]:
config = load_config("config_graph.yaml")

env, solvers, trainer, config = create_graph_mfg_from_config(config)

# Training loop
losses = []

Target Device: cpu


c:\Uzh\thesis\msc_code\thesis\environment\environment.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.A = torch.tensor(adjacency_matrix, dtype=torch.float32, device=device)


In [4]:
flows, final_flows, policies = solve_multigroup(env, solvers)

tensor([[[1.0000, 5.0052, 5.5000, 1.0000],
         [1.0000, 1.0000, 1.0000, 5.5000],
         [1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000]],

        [[0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000]],

        [[0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000]],

        [[0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000]],

        [[0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000]],

        [[0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000],
 

In [5]:
final_flows

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 9.8142e-01, 1.8579e-02, 0.0000e+00],
        [0.0000e+00, 9.8142e-01, 1.8579e-02, 0.0000e+00],
        [0.0000e+00, 9.8142e-01, 1.8579e-02, 0.0000e+00],
        [0.0000e+00, 9.8142e-01, 1.8579e-02, 0.0000e+00],
        [0.0000e+00, 9.8142e-01, 1.8579e-02, 0.0000e+00],
        [0.0000e+00, 9.8142e-01, 1.8579e-02, 0.0000e+00],
        [1.4441e-08, 2.6829e-10, 1.4172e-08, 1.0000e+00],
        [1.4441e-08, 2.6829e-10, 1.4172e-08, 1.0000e+00],
        [2.0853e-16, 1.4441e-08, 3.8743e-18, 1.0000e+00],
        [2.0853e-16, 1.4441e-08, 3.8743e-18, 1.0000e+00],
        [2.0853e-16, 2.0853e-16, 2.0853e-16, 1.0000e+00],
        [2.0853e-16, 2.0853e-16, 2.0853e-16, 1.0000e+00],
        [6.0226e-24, 2.0853e-16, 3.0113e-24, 1.0000e+00],
        [6.0226e-24, 2.0853e-16, 3.0113e-24, 1.0000e+00],
        [3.0113e-24, 6.0226e-24, 3.0113e-24, 1.0000e+00],
        [3.0113e-24, 6.0226e-24, 3.0113e-24, 1.0000e+00],
        [1.304

In [6]:
flows[0,:,:,0]

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 9.8142e-01, 1.8579e-02, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.4441e-08, 2.6829e-10, 1.4172e-08, 1.8579e-02],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.8579e-02],
        [2.0853e-16, 1.4441e-08, 3.8743e-18, 1.8579e-02],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.8579e-02],
        [2.0853e-16, 2.0853e-16, 2.0853e-16, 1.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [6.0226e-24, 2.0853e-16, 3.0113e-24, 1.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [3.0113e-24, 6.0226e-24, 3.0113e-24, 1.0000e+00],
        [0.000